<a href="https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import os
import subprocess
import pandas as pd
import numpy as np

# Get the starter repository if needed
if not os.path.exists("flyrank-ml-internship-starter"):
    subprocess.run([
        "git", "clone",
        "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    ], check=True)

df = pd.read_csv(
    "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
)

features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "search_volume"
]

X = df[features].replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

print("Feature vector shape:", X.shape)
print("Features used:")
print(features)

X.head()

Feature vector shape: (30000, 7)
Features used:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count', 'search_volume']


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,search_volume
0,187,20,3803,10.6,0.76,3221.0,10.0
1,445,25,15320,20.3,0.05,2481.0,90.0
2,141,20,12581,36.5,0.09,3515.0,0.0
3,463,22,11751,6.2,0.49,0.0,10.0
4,263,14,19140,44.0,0.13,2803.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The features describe information that can be observed about a webpage before making a refresh-priority decision.

- content_age_days: how old the content is. Missing values are filled with 0. This can be known before prediction.
- days_since_last_update: how many days have passed since the page was updated. Missing values are filled with 0. This can be known before prediction.
- impressions_90d: search impressions over the previous 90 days. Missing values are filled with 0. This is historical information available before prediction.
- avg_position: average search position. Missing values are filled with 0. This is historical search information available before prediction.
- ctr: click-through rate. Missing values are filled with 0. This is historical search information available before prediction.
- word_count: approximate number of words on the page. Missing values are filled with 0. This can be known before prediction.
- search_volume: estimated search volume. Missing values are filled with 0. This is used as an available search signal.

These features are numerical, so no categorical encoding is needed for this feature vector.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked the features for possible leakage. I will not use trend_pct as a feature because it is directly related to the trend outcome used to create the declining label. I will also avoid product decision flags or any information that becomes available only after the outcome. The model should use information that was available before the prediction.

In [2]:
# Check for suspicious outcome-related fields
possible_leakage = [
    "trend_pct",
    "trend_direction"
]

for col in possible_leakage:
    print(col, "exists:", col in df.columns)

print("\nFeatures actually used:")
print(features)

trend_pct exists: True
trend_direction exists: True

Features actually used:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count', 'search_volume']


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

I excluded the following fields:

- trend_pct: excluded because it is directly related to the outcome and could cause leakage.
- trend_direction: excluded as a model feature because it is used to define the declining label.
- Any product decision flags: excluded because they may already contain a previous decision or recommendation.
- Client names, URLs, or private query information: excluded to protect privacy and because they are not needed for this analysis.



In [3]:
excluded = [
    "trend_pct",
    "trend_direction"
]

print("Excluded fields:")
for col in excluded:
    print("-", col)

print("\nFinal features:")
for col in features:
    print("-", col)

Excluded fields:
- trend_pct
- trend_direction

Final features:
- content_age_days
- days_since_last_update
- impressions_90d
- avg_position
- ctr
- word_count
- search_volume


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.